# Data Review

**Goal:**  
Review each cleaned dataset on its own to understand what information it contains and decide which columns should be used later in the merge.

**Plan:**
1. Review ACS data
2. Review FMR data
3. Review HPI data
4. Review HADS data
5. Compare the year and county coverage across files
6. Identify the key columns needed for merging

In [5]:
from pathlib import Path
import pandas as pd
import numpy as np

### Step 1: Review ACS Data

This step loads the cleaned ACS dataset and checks what information it contains.

In [6]:
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

clean_path = project_root / "clean_data"

acs = pd.read_csv(
    clean_path / "ACS_clean.csv",
    dtype={
        "state_fips": str,
        "county_fips": str,
        "county_fips_full": str
    }
)

acs.head()

,county_name,total_occupied_units,owner_occupied_total,owner_occupied_hh_15_24,owner_occupied_hh_25_34,owner_occupied_hh_35_44,owner_occupied_hh_45_54,owner_occupied_hh_55_59,owner_occupied_hh_60_64,owner_occupied_hh_65_74,...,renter_occupied_hh_35_44,renter_occupied_hh_45_54,renter_occupied_hh_55_59,state_fips,county_fips,year,county_clean,county_fips_full,owner_rate,renter_rate
0,"Atlantic County, New Jersey",101584,68970,980,7452,14735,16400,6754,5659,7839,...,7108,6368,1624,34,001,2005,Atlantic,34001,0.678946,0.321054
1,"Bergen County, New Jersey",332223,225175,763,13773,50079,54557,24781,20251,28909,...,25357,20069,8337,34,003,2005,Bergen,34003,0.677783,0.322217
2,"Burlington County, New Jersey",164046,129201,1360,14486,28484,34191,11863,10931,14120,...,6761,5952,2318,34,005,2005,Burlington,34005,0.787590,0.212410
3,"Camden County, New Jersey",190659,130576,1141,14690,26698,33575,14054,10884,14414,...,13478,10447,3196,34,007,2005,Camden,34007,0.684867,0.315133
4,"Cape May County, New Jersey",43616,34101,257,2457,5958,7586,3447,3174,5332,...,1614,1190,680,34,009,2005,Cape May,34009,0.781846,0.218154


### ACS Columns to Keep for the Merge

For the merged dataset, I want to keep the ACS columns that help explain ownership and renting patterns by county and year.

Columns to keep:

- `county_fips_full`: merge key for county
- `county_clean`: county name
- `year`: merge key for year
- `total_occupied_units`: total occupied housing units
- `owner_occupied_total`: total owner-occupied units
- `renter_occupied_total`: total renter-occupied units
- `owner_rate`: percent of occupied units that are owner-occupied
- `renter_rate`: percent of occupied units that are renter-occupied
- owner age columns: owner-occupied units by householder age group
- renter age columns: renter-occupied units by householder age group

Columns that do not need to be used in the merge:

- `county_name`: longer raw county name
- `state_fips`: already included inside the full county FIPS code
- `county_fips`: only the 3-digit county code, not the full merge key

### Step 2: Review FMR Data

In [7]:
fmr = pd.read_csv(
    clean_path / "FMR_clean.csv",
    dtype={"county_fips": str}
)

fmr.head()

,county_fips,county,year,fmr_0br,fmr_1br,fmr_2br,fmr_3br,fmr_4br,fmr_percentile
0,34001,Atlantic,1993,451,552,647,808,909,45
1,34003,Bergen,1993,645,783,929,1161,1301,45
2,34005,Burlington,1993,444,538,634,793,889,45
3,34007,Camden,1993,444,538,634,793,889,45
4,34009,Cape May,1993,451,552,647,808,909,45


### FMR Columns to Keep for the Merge

For the merged dataset, I want to keep the FMR columns that show rent costs by county and year.

Columns to keep:

- `county_fips`: merge key for county
- `county`: county name
- `year`: merge key for year
- `fmr_0br`: fair market rent for a 0-bedroom unit
- `fmr_1br`: fair market rent for a 1-bedroom unit
- `fmr_2br`: fair market rent for a 2-bedroom unit
- `fmr_3br`: fair market rent for a 3-bedroom unit
- `fmr_4br`: fair market rent for a 4-bedroom unit

Columns that do not need to be used in the merge:

- `fmr_percentile`: describes the percentile HUD used to calculate FMR, but it is not a rent amount

### Step 3: Review HPI Data

In [9]:
hpi = pd.read_csv(
    clean_path / "hpi_at_county_1993-2013.csv",
    dtype={"FIPS code": str}
)

hpi.head()

,State,County,FIPS code,Year,Annual Change (%),HPI,HPI with 1990 base,HPI with 2000 base
0,NJ,Atlantic,34001,1993,0.01,349.87,99.97,85.51
1,NJ,Atlantic,34001,1994,-2.20,342.16,97.76,83.63
2,NJ,Atlantic,34001,1995,-1.04,338.60,96.75,82.76
3,NJ,Atlantic,34001,1996,0.93,341.77,97.65,83.53
4,NJ,Atlantic,34001,1997,3.19,352.66,100.76,86.20


### HPI Columns to Keep for the Merge

For the merged dataset, I want to keep the HPI columns that show home price changes by county and year.

Columns to keep:

- `FIPS code`: merge key for county
- `County`: county name
- `Year`: merge key for year
- `Annual Change (%)`: percent change in house prices from the previous year
- `HPI`: House Price Index value. This shows how home prices changed over time.

Columns that do not need to be used in the merge:

- `State`: all rows are New Jersey
- `HPI with 1990 base`: extra comparison column using 1990 as the base year
- `HPI with 2000 base`: extra comparison column using 2000 as the base year

### Step 3: Review CPS Data


In [10]:
cps = pd.read_csv(clean_path / "CPS_1993_2013.csv")

cps.head()

,Merge,State,Sub/Urb,HH_Size,Tenure,HH_Income,Supp_Weight,Relationship,Weight,Age,Education,Marital,Person_Income,Year,County
0,10872,22,4,1,2,7753,147225,2,1463.03,63,36,5,7753,1994,NaN
1,10871,22,4,2,2,17175,134162,1,1333.48,65,34,7,12317,1994,NaN
2,10870,22,4,1,2,5610,84994,2,1717.62,61,32,4,5610,1994,NaN
3,10869,22,4,1,2,1680,130128,2,1420.01,56,37,6,1680,1994,NaN
4,10867,22,4,2,1,24884,113212,1,1124.33,49,39,4,22367,1994,NaN


In [11]:
cps.shape

(35874, 15)